In [ ]:
# Import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [ ]:
device = torch.device('cuda')

In [ ]:
# DATA
df = pd.read_csv("/content/fashion-mnist_train.csv")
X = df.iloc[:5000,1:].values
y = df.iloc[:5000,0].values
# print(type(X))
# print(type(y))
# print(y.shape[0])

X = torch.tensor(X)
y = torch.tensor(y)
print(type(X))
print(type(y))
print(X.shape)
print(y.shape)
# X = X.numpy()
# y = y.numpy()
# print(type(X))
# print(type(y))


<class 'torch.Tensor'>
<class 'torch.Tensor'>
torch.Size([5000, 784])
torch.Size([5000])


In [ ]:
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size= 0.2, random_state=42)
print(train_x.shape)


torch.Size([4000, 784])


In [ ]:
# Dataset and Dataloader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels
  def __len__(self):
    # return self.labels.shape[0]
    return len(self.features)
  def __getitem__(self, index):
    return (self.features[index], self.labels[index])

train_ds = CustomDataset(train_x, train_y)
test_ds = CustomDataset(test_x, test_y)
train_dl = DataLoader(train_ds, batch_size=32,  shuffle=True)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=True)


In [ ]:
# for i in range(1):
#   for batch_features, batch_labels in train_dl:
#     print(batch_features.shape)
#     print(batch_labels.shape)

In [ ]:
# Model definition
class SimpleANN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,10)
        #Auto Softmax will be applied
    )
  def forward(self, X):
    out = self.model(X)
    return out

In [ ]:
# Parameters
learning_rate = 10e-3
epochs = 100
loss_fn = nn.CrossEntropyLoss()

In [85]:
# Model Training
model = SimpleANN(train_x.shape[1])
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr= learning_rate)
for epoch in range(epochs):
  loss = 0
  for batch_features, batch_labels in train_dl:
    batch_features, batch_labels = batch_features.float().to(device), batch_labels.to(device)
    # print(batch_labels)
    # Forward pass
    y_pred = model(batch_features)
    # Loss Calc
    loss = loss_fn(y_pred, batch_labels)
    # print(loss)
    # Grad zero
    optimizer.zero_grad()
    # Backward pass
    loss.backward()
    # Weight updation
    optimizer.step()
    # loss for the epoch
    loss += loss
  loss /= batch_features.shape[0]
  print(f"For Epoch {epoch} loss = {loss}")

For Epoch 0 loss = 0.14466607570648193
For Epoch 1 loss = 0.1433076411485672
For Epoch 2 loss = 0.1435488760471344
For Epoch 3 loss = 0.14437930285930634
For Epoch 4 loss = 0.14417581260204315
For Epoch 5 loss = 0.14399373531341553
For Epoch 6 loss = 0.14442405104637146
For Epoch 7 loss = 0.1435968279838562
For Epoch 8 loss = 0.14383746683597565
For Epoch 9 loss = 0.14401932060718536
For Epoch 10 loss = 0.14367198944091797
For Epoch 11 loss = 0.14418987929821014
For Epoch 12 loss = 0.1437223106622696
For Epoch 13 loss = 0.14470335841178894
For Epoch 14 loss = 0.14403681457042694
For Epoch 15 loss = 0.14394697546958923
For Epoch 16 loss = 0.14381654560565948
For Epoch 17 loss = 0.1437247395515442
For Epoch 18 loss = 0.14398781955242157
For Epoch 19 loss = 0.1441294550895691
For Epoch 20 loss = 0.14410418272018433
For Epoch 21 loss = 0.14387500286102295
For Epoch 22 loss = 0.14431649446487427
For Epoch 23 loss = 0.14432406425476074
For Epoch 24 loss = 0.14365267753601074
For Epoch 25 los

In [88]:
# Model Evaluation
model.eval()

with torch.no_grad():
    correct = 0
    total = 0
    for batch_features, batch_labels in test_dl:
      batch_features, batch_labels = batch_features.float().to(device), batch_labels.to(device)
      # Forward pass
      y_pred = model(batch_features)
      # print(y_pred)
      _,pos_Val = torch.max(y_pred, 1)
      total += total+batch_features.shape[0]
      correct += (pos_Val == batch_labels).sum().item()

    print(correct/total)

6.330083126918796e-10
